In [0]:
from pyspark.sql import functions as F

silver_df = spark.readStream.table("crp_arb_dev.silver.crypto_data_24h")

gold_24h_df = silver_df \
    .withWatermark("event_timestamp", "1 day") \
    .groupBy(
        F.window("event_timestamp", "1 day"),
        "ticker"
    ).agg(
        F.first("open").alias("open"),
        F.last("close").alias("close"),
        F.max("high_price").alias("day_high"),
        F.min("low_price").alias("day_low"),
        F.sum("trading_volume").alias("day_volume")
    ) \
    .select(
        F.col("window.start").cast("date").alias("date"),
        "ticker", "open", "close", "day_high", "day_low", "day_volume"
    ) \
    .withColumn("daily_perf_pct", F.round(((F.col("close") - F.col("open")) / F.col("open")) * 100, 2))

container_path = "abfss://crp-arb-cnt@sadlsdev.dfs.core.windows.net"
query_24h = gold_24h_df.writeStream \
    .format("delta") \
    .outputMode("complete") \
    .option("checkpointLocation", f"{container_path}/_checkpoints/gold_1d_window") \
    .option("path", f"{container_path}/gold/crypto_data_1d_window") \
    .trigger(availableNow=True) \
    .toTable("crp_arb_dev.gold.crypto_data_24h_window")
    
query_24h.awaitTermination()